In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ L = -y\ln(p)-(1-y)\ln(1-p) $$
$$ \frac{\partial L}{\partial p} = -y\frac{1}{p} + (1-y)\frac{1}{1-p} = \frac{p-y}{p(1-p)} $$

$$ \diamond \diamond \diamond $$

$$ \text{Let } p = sigmoid(z) $$
$$ \frac{\partial L}{\partial z} = \frac{\partial L}{\partial p} \cdot \frac{\partial p}{\partial z} = \frac{p-y}{p(1-p)} \cdot p(1-p) = p-y $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from common import assert_eq, T # type: ignore


def binary_cross_entropy(p: tr.Tensor, y: tr.Tensor, reduction:str = "mean") -> tr.Tensor:
    p = p.clamp(0.0001, 0.9999)
    bce = - (y * p.log() + (1 - y) * (1 - p).log())
    
    if reduction == "mean":
        return bce.mean()
    else:
        return bce


class BinaryCrossEntropyFunction(autograd.Function):
    @staticmethod
    def forward(ctx, p: tr.Tensor, y: tr.Tensor) -> tr.Tensor:
        p = p.clamp(0.0001, 0.9999)
        ctx.save_for_backward(p, y)
        return binary_cross_entropy(p, y)

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, None]:
        (p, y) = ctx.saved_tensors

        df_dp = (p - y) / (p * (1 - p))

        # If `binary_cross_entropy` is averaged by `mean`,
        # autograd sets `dF_df = 1/N` automatically.

        return (dF_df * df_dp, None)
   

# `BinaryCrossEntropyFunction` implements the actual operator.
# `BinaryCrossEntropy(Module)` is just a wrapper for nicer API.
class BinaryCrossEntropy(nn.Module):
    def forward(self, p: tr.Tensor, y: tr.Tensor) -> tr.Tensor:
        return BinaryCrossEntropyFunction.apply(p, y)


def test_binary_cross_entropy():
    tr.manual_seed(0)

    samples = 10
    p = tr.rand(samples, 1, requires_grad=True)
    y = tr.randint(0, 2, (samples, 1)).float()

    p1 = p.clone().detach().requires_grad_(True)
    y1 = BinaryCrossEntropy()(p1, y)
    y1.backward()

    p2 = p.clone().detach().requires_grad_(True)
    y2 = nn.BCELoss()(p2, y)
    y2.backward()

    assert_eq(y1.item(), y2.item(), atol=1e-4)
    assert_eq(p1.grad, p2.grad, atol=1e-4)


if __name__ == "__main__":
    test_binary_cross_entropy()